## Wczytanie danych i bibliotek

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv('../data/Hotel Reservations.csv')

# 2. Przetwarzanie danych

### Usunięcie kolumny Booking_ID

In [ ]:
df = df.drop('Booking_ID', axis = 1)
df.head()

Zmienna ta nie wnosiłą nic wartościowego pod względem implementacji modelu.

### Kodowanie zmiennej docelowej

In [ ]:
target = {'Canceled': 1, 'Not_Canceled': 0} # Zmiana booking_status na liczbe
df['booking_status'] = df['booking_status'].replace(target)
print("Canceled = 1, Not_Canceled = 0")
df.describe()

### Dodanie nowych kolumn

In [ ]:
df['is_free_room'] = (df['avg_price_per_room'] == 0).astype(int)

W EDA wyszło, że dla ceny pokoju wynoszącej 0, bardzo rzadko rezerwacja jest anulowana. Z tego względu tworzona jest kolumna `is_free_room`, która binarnie mówi o tym, czy pokój był darmowy.

In [ ]:
total_nights = df['no_of_week_nights'] + df['no_of_weekend_nights']

df['is_zero_nights'] = (total_nights == 0).astype(int)

zero_nights_count = df['is_zero_nights'].sum()
print(f"Liczba rezerwacji z liczbą nocy równą 0: {zero_nights_count}")

zero_nights_rates = df.groupby('is_zero_nights')['booking_status'].mean() * 100

print("\nWskaźnik anulacji (%):")
print(f"Standardowy pobyt (więcej niż 0 nocy): {zero_nights_rates[0]:.2f}%")
if zero_nights_count > 0:
    print(f"Pobyt 0-dniowy (is_zero_nights = 1):   {zero_nights_rates[1]:.2f}%")

In [ ]:

print("Ilu gości spełnia oba warunki na raz?")
crosstab_result = pd.crosstab(
    df['is_zero_nights'], 
    df['is_free_room'], 
    margins=True,
    rownames=['Day Use (0 nocy)'], 
    colnames=['Darmowy (0 zł)']
)
display(crosstab_result)

korelacja = df[['is_zero_nights', 'is_free_room']].corr().iloc[0, 1]
print(f"\nWspółczynnik korelacji wynosi: {korelacja:.4f}")

### Podział danych na zbiór treningowy, walidacyjny i testowy w proporcjach 80/10/10

In [ ]:
X = df.drop('booking_status', axis = 1)
y = df['booking_status']

test_size = 0.10
val_size = 0.10
X_train_val, X_test, y_train_val, y_test = train_test_split( # 10%
    X, y, test_size=test_size, stratify=y, random_state=123)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_size/(1 - test_size), stratify=y_train_val, random_state=123)

### Kodowanie zmiennych kategorycznych

In [ ]:
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeryczne cechy:", num_features)
print("Kategoryczne cechy:", cat_features)

In [ ]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

data_pipeline.fit(X_train)

feature_names = (
    pd.Index(data_pipeline.named_steps['preprocessor'].get_feature_names_out())
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

X_train_processed = pd.DataFrame(data_pipeline.transform(X_train),
                                columns=feature_names)

X_val_processed = pd.DataFrame(data_pipeline.transform(X_val),
                                columns=feature_names)

X_test_processed = pd.DataFrame(data_pipeline.transform(X_test),
                                columns=feature_names)

### Zapisanie przetworzonych danych

In [ ]:
import joblib, os

joblib.dump({
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    'X_train_processed': X_train_processed,
    'X_val_processed':   X_val_processed,
    'X_test_processed':  X_test_processed,
    'feature_names': feature_names,
    'df': df,
}, '../data/processed_data.pkl')